In [1]:
%pip install pandas numpy scikit-learn xgboost

   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   - -------------------------------------- 3.1/124.9 MB 26.2 MB/s eta 0:00:05
   -- ------------------------------------- 7.9/124.9 MB 22.1 MB/s eta 0:00:06
   ----- ---------------------------------- 15.7/124.9 MB 28.3 MB/s eta 0:00:04
   ------- -------------------------------- 22.8/124.9 MB 29.4 MB/s eta 0:00:04
   --------- ------------------------------ 29.4/124.9 MB 30.0 MB/s eta 0:00:04
   ----------- ---------------------------- 34.9/124.9 MB 29.1 MB/s eta 0:00:04
   ------------- -------------------------- 41.9/124.9 MB 30.6 MB/s eta 0:00:03
   ------------- -------------------------- 43.3/124.9 MB 26.5 MB/s eta 0:00:04
   --------------- ------------------------ 49.8/124.9 MB 27.1 MB/s eta 0:00:03
   ----------------- ---------------------- 56.1/124.9 MB 27.3 MB/s eta 0:00:03
   -------------------- ------------------- 63.2/124.9 MB 28.0 MB/s eta 0:00:03
   ---------------------- ----------------- 69.2/12

In [15]:
import pandas as pd
import numpy as np

In [12]:
import os

# Walk through the dataset directory to find the path to the text files
for root, dirs, files in os.walk('dataset_fog_release'):
    txt_files = [f for f in files if f.endswith('.txt')]
    if txt_files:
        print(f"Found text files in: {root}")
        print(f"Sample files: {txt_files[:3]}")
        break

Found text files in: dataset_fog_release\dataset
Sample files: ['S01R01.txt', 'S01R02.txt', 'S02R01.txt']


In [19]:
import numpy as np
import pandas as pd

# 1. Define parameters for the sliding window
window_size = 128  # At 64Hz, 128 rows = exactly 2 seconds of data
step_size = 64     # 64 rows = 1 second overlap (50% overlap)

# List to hold our newly engineered features
extracted_features = []

print(f"Creating sliding windows (Size: {window_size} rows)...")

# 2. Loop through the dataset in chunks (FIXED: added +1 to include the last window)
for i in range(0, len(df_clean) - window_size + 1, step_size):
    
    # Grab the current 2-second window of data
    window = df_clean.iloc[i : i + window_size]
    
    # 3. Determine the label for this window 
    # (IMPROVED: Require at least 25% of the window to be a freeze to avoid extreme smearing)
    if window['Label'].mean() >= 0.25:
        label = 1
    else:
        label = 0
        
    # 4. Extract statistical features for Ankle_X
    ankle_x_data = window['Ankle_X']
    
    feature_dict = {
        'Ankle_X_Mean': np.mean(ankle_x_data),
        'Ankle_X_Std': np.std(ankle_x_data),                # Standard Deviation
        'Ankle_X_Variance': np.var(ankle_x_data),           # Variance (Kinematic energy)
        'Ankle_X_RMS': np.sqrt(np.mean(ankle_x_data**2)),   # Root Mean Square
        'Label': label
    }
    
    # 5. Append the engineered features to our lists
    extracted_features.append(feature_dict)

# 6. Convert the list of dictionaries back into a clean Pandas DataFrame
df_features = pd.DataFrame(extracted_features)

print(f"Feature Extraction Complete!")
print(f"Original raw rows: {len(df_clean)}")
print(f"New temporal windows: {len(df_features)}")

# 7. Preview the new highly-engineered dataset
df_features.head()

Creating sliding windows (Size: 128 rows)...


NameError: name 'df_clean' is not defined